# seq2seq 模式介绍



<img src="./markdown-note/images/img_18.png" width="500"/>




图中表示的是一个中文到英文的翻译：欢迎来北京→welcome to BeiJing。编码器首先处理中文输入"欢迎来北京”，通过GRU模型获得每个时间步的输出张量，最后将它们拼接成一个中间语义张量c；接着解码器将使用这个中间语义张量c以及每一个时间步的隐层张量，逐个生成对应的翻译语言

早期在解决机器翻译这一类seq2seq问题时，通常采用的做法是利用一个编码器（Encoder）和一个解码器（Decoder）构建端到端的神经网络模型，但是基于编码解码的神经网络存在两个问题：

问题1：如果翻译的句子很长很复，比如直接一篇文章输进去，模型的计算量很大，并且模型的准确率下降严重。

问题2：在翻译时，可能在不同的语境下，同一个词具有不同的含义，但是网络对这些词向量并没有区分度，没有考虑词与词之间的相关性，导致翻译效果比较差。

主要就是每次都用这个中间语义张量 C 来生成翻译语言 ，但是这个中间语义张量 C 并没有考虑词与词之间的相关性，也没有侧重点

比如机器翻译任务，输入source为：Tom chase Jerry，输出target为：“汤姆”，“追逐”，“杰瑞”。在翻译"Jerry"这个词为中文的时候，普通Encoder-Decoder框架中，source里的每个单词对翻译目标单词"杰瑞"贡献是相同的，很明显这里不太合理，显然"Jerry"对于翻译成"杰瑞"更重要。

# 注意力机制

通俗来讲就是对于模型的每一个输入顶，可能是图片中的不同部分，或者是语句中的某个单词分配一个权重，这个权重的大小就代表了我们希望模型对该部分一个关注程度。这样一来，通过权重大小来模拟人在处理信息的注意力的侧重，有效的提高了模型的性能，并且一定程度上降低了计算量。

深度学习中的注意力机制通常可分为三类：软注意（全局注意）、硬注意（局部注意）和自注意 (内注意）

- 软注意机制(Soft/GlobalAttention:对每个输入项的分配的权重为0-1之间，也就是某些部分关注的多一点，某些部分关注的少一点，因为对大部分信息都有考虑，但考虑程度不一样，所以
相对来说计算量比较大。

- 硬注意机制（Hard/LocalAttention,[了解即可])：对每个输入项分配的权重非o即1，和软注意不同，硬注意机制只考虑那部分需要关注，哪部分不关注，也就是直接舍弃掉一些不相关项。优
势在于可以减少一定的时间和计算成本，但有可能丢失掉一些本应该注意的信息。

- 自注意机制（Self/IntraAttention)：对每个输入项分配的权重取决于输入项之间的相互作用，即通过输入项内部的"表决"来决定应该关注哪些输入项。和前两种相比，在处理很长的输入
时，具有并行计算的优势。

## encoder-decoder 模式加入注意力机制

普通的Encoder和Decoder 框架如下图所示：

<img src="./markdown-note/images/img_19.png" width="500"/>">

上图图例可以把它看作由一个句子（或篇章）生成另外一个句子（或篇章）的通用处理模型。
对于句子对，我们的目标是给定输入句子Source，期待通过Encoder-Decoder框架来生成目标句子Target。
Source和Target可以是同一种语言，也可以是两种不同的语言。而Source和Target分别由各自的单词序列构成：

Source =(X1,X2...Xm)

Target = (yn, y2...yn)

那么中间语义张量C可以表示为：

C = F(X1,X2...Xm)


对于解码器Decoder来说，其任务是根据句子Source的中间语义表示C和之前已经生成的历史信息,y1, y_2..y_i-1来生成i时刻要生成的单词y_i


y_i = G(C,y_1, y_2..y_i-1)

由此可得

y_0 = f(C)

y_1 = f(C,y_0)

y_2 = f(C,y_0,y_1)

其中f 是 Decoder的非线性变换函数。从这里可以看出，在生成目标句子的单词时，不论生成哪个单词，它们使用的输入句子Source的语义编码C都是一样的，没有任何区别。而语义编码C又是通过对source经过Encoder编码产生的，因此对于target中的任何一个单词，source中任意单词对某个目标单词y_i来说影响力都是相同的，这就是为什么说上面的模型没有体现注意力的原因。

接下来我们给这个模型加入注意力机制

举例说明，如何添加Attention:
比如机器翻译任务，输入source为：Tom chase Jerry， 输出target为：“汤姆”，“追逐”，“杰瑞”。在翻译”Jerry”为 "杰瑞" 的时候，普通Encoder-Decoder框架中，source里的每个单词对翻译目标单词“杰瑞“贡献是相同的，很明显这里不太合理，显然”Jerry”对于翻译成"杰瑞”更重要。

如果引l入Attention模型，在生成”杰瑞"的时候，应该体现出英文单词对于翻译当前中文单词不同的影响程度，比如给出类似下面一个概率分布值：（Tom,0.3）（Chase,0.2)（Jerry,0.5).每个英文单词的概率代表了翻译当前单词“杰瑞”时，注意力分配模型分配给不同英文单词的注意力大小。

因此，基于上述例子所示，对于target中任意一个单词都应该有对应的source中的单词的注意力分配概率

而且，由于注意力模型的加入，原来在生成target单词时候的中间语义C 就不再是固定的，而是会根据注意力概率变化的C，加入了注意力模型的Encoder-Decoder框架就变成了

<img src="./markdown-note/images/img_20.png" width="500"/>">


即生成目标句子单词的过程成了下面的形式：

y1= f1(C1)

y2 = f1(C2,y1)

y3 = f1(C3,y1,y2)

而每个Ci可能对应着不同的源语句子单词的注意分配概率分布，比如对于上面的英汉翻译来说，其对应的信息可能如下：

C_Tom = g(0.6 * f2(Tom), 0.2 * f2(Chase), 0.2 * f2(Jerry))

C_chase = g(0.2 * f2(Tom), 0.7 * f2(Chase), 0.1 * f2(Jerry))

C_Jerry = g(0.3 * f2(Tom), 0.2 * f2(Chase), 0.5 * f2(Jerry))

f2函数代表Encoder对输入英文单词的某种变换函数，比如如果Encoder是用的RNN模型的话，这个f2函数的结果往往是某个时刻输入后隐层节点的状态值；

g代表Encoder根据单词的中间表示合成整个句子中间语义表示的变换函数，一般的做法中，g函数就是对构成元素加权求和，即下列公式


<img src="./markdown-note/images/img_22.png" width="500"/>"

<img src="./markdown-note/images/img_23.png" width="500"/>">


## 注意力的概率计算过程

其实Attention机制可以看作，Target中每个单词是对Source每个单词的加权求和，而权重是Source中每个单词对Target中每个单词的重要程度。因此，Attention的本质思想会表示成下图：

<img src="./markdown-note/images/img_30.png" width="1200"/>"

我们把上图的过程 改为公式表示如下：

<img src="./markdown-note/images/img_28.png" width="500"/>


在上面的这个翻译的例子里面我们KEY 和 VALUE 都是h_i  但是需要知道的是 不一定在所有场景中这两者都是相同的 **也就是说参与相似度计算最终得到权重的 Key 和 参与加权计算注意力的 Value 可以是不同的**


在更高级的模型（比如 Transformer）里，它们是可以分开的，这是为了让模型更灵活、更强大。

我们可以把这个过程理解为：

Key 负责 “对齐”：它的任务是和 Query 计算相似度，决定 “我应该关注哪里”。它更像是一个 “注意力探测器”。

Value 负责 “提供信息”：它的任务是被加权求和，提供 “我关注到的内容”。它更像是一个 “信息存储器”。

当我们把 Key 和 Value 分开时，模型就可以学到：

用不同的方式来表示 “我是谁”（Key）和 “我有什么”（Value）。

这让模型的表达能力大大增强，可以处理更复杂的任务。

> 这里说的注意力计算方式叫做「加性注意力」（Bahdanau Attention）transformer 里面的注意力计算方式与此不同，而是直接让 Q 和 K 做矩阵乘法（点积）



注意力机制的一般规则：

它需要三个指定的输入Q(query), K(key), V(value), 然后通过计公式得到注意力的结果,这个结果代表query在key和value作用下的注意力表示.当输入的Q=K=V时，称作自注意力计算规则；当Q、K、V不相等时称为一般注意力计算规则，自注意力等到介绍transformer的时候在介绍

<img src="./markdown-note/images/img_32.png" width="500"/>


## 以翻译案例为演示 详细解释QKV

<img src="./markdown-note/images/attention1.png" width="1500"/>


## 注意力计算实操

In [8]:
import torch

# 回忆一下二维矩阵的乘法运算
a = torch.randn(3, 4)
b = torch.randn(4, 5)
c1 = torch.matmul(a, b)
c2 = a @ b
c3 = torch.mm(a, b)
# c3= a*b  # 点乘 无法执行 a和b的形状不一样
print(c1.shape)
print(c2.shape)
print(c3.shape)

# 介绍下三维矩阵的运算 bmm运算
# 参数1 的形状是 (b,m,n) 参数2的形状是 (b,n,p) 则最终结果为 (b,m,p)
a = torch.randn(2, 3, 4)
b = torch.randn(2, 4, 5)
c = torch.bmm(a, b)
print(c.shape)

a = torch.randn(1, 3, 4)
b = torch.randn(2, 4, 5)
# bmm 运算要求第一个维度一定要相等 不等的话是无法执行的
#c = torch.bmm(a,b)

c2 = torch.matmul(a, b)
# torch.Size([2, 3, 5])  使用矩阵乘法API是可以做多维矩阵运算的
# 这里因为a的第一维是1 可以应用广播，如果a的第一维不是1 也不能运算
print(c2.shape)

# mm和bmm都要求形状必须一样 没有广播机制调整
# c2 = torch.mm(a,b)


a = torch.randn(3, 4)
b = torch.randn(2, 4, 5)
# a 虽然少一维  依然可以做运算 结果为(2,3,5) 相当于广播运算
c2 = torch.matmul(a, b)


torch.Size([3, 5])
torch.Size([3, 5])
torch.Size([3, 5])
torch.Size([2, 3, 5])
torch.Size([2, 3, 5])


## 加性注意力工程实现（重点）

在我们之前的翻译案例解释的过程中 我们发现计算Q和K的相似度貌似很简单 但是真实落地到代码实现的时候 由于Q和K的维度可能不一样，比如编码器的隐藏层维度设置为512 解码器的隐藏层维度设置为128  这时候他两计算相似度就比较困难了  加性注意力的代码实现的时候计算规则是这样的：

（1） Q 与 每一个Ki 做拼接 然后经过一个线性层处理（这个线性层一般用来做降维 避免过多的特征维度），再经过一个tanh激活函数处理，得到一个[-1,1] 的输出结果

（2） 将第一步的输出结果 再经过一个线性层处理 得到Q和Ki的相似度 这一步的结果主要就是要生成一个相似度分分数标量

（3） 将所有的得分经过softmax处理 得到一个权重向量，得到注意力权重，用这个权重对V进行加权求和得到注意力表示，这个注意力表示将用来预测下一个词


还是以「生成 Welcome 时，Q₁匹配 K₁（欢迎）、K₂（来）、K₃（北京）」为例，

**步骤 1：Q 与每个 Ki 拼接→线性层→tanh**

- Q₁（128 维）分别和 K₁（128）、K₂（128）、K₃（128）拼接 → 得到 3 个 256 维向量；
- 每个 256 维向量经过线性层降维到 128 维 → 压缩特征，聚焦关键信息；
- 128 维向量经过 tanh → 输出 3 个 [-1,1] 的 128 维特征向量（分别对应 Q₁和 K₁/K₂/K₃的匹配特征）。

**步骤 2：线性层生成标量相似度得分**

- 3 个 128 维特征向量分别经过线性层 → 压缩成 3 个标量（比如：90 分、5 分、5 分）；
这 3 个标量就是 Q₁和 K₁/K₂/K₃的「相似度得分」。


**步骤 3：Softmax→权重向量→加权 Vi 得到注意力表示**
- 3 个得分经过 Softmax → 得到权重向量：[0.9, 0.05, 0.05]（这是注意力权重）；
- 用权重向量对 V₁（欢迎）、V₂（来）、V₃（北京）加权求和：
**上下文向量 = 0.9×V₁ + 0.05×V₂ + 0.05×V₃；**

- 这个上下文向量才是注意力层最终输出的「注意力表示」，解码器拿着它生成 Welcome。这个上下文向量就相当于我们之前画的图中的C1

**步骤 4：将Q1 和 C1 作为解码器的输入，来生成welcome**
- 首先将Q1和C1拼接，也就是做信息合并，然后经过一个线性层处理 作为GRU的输入（输入之前需要把二维向量转为三维向量（序列长度、batch_size、特征维度）））
- 注：另一个输入就是GRU上一个时间步的输出隐藏状态


以上过程用图示表示如下：

<img src="./markdown-note/images/img_35.png" width="1000"/>


In [10]:
# 导入PyTorch核心库：nn是神经网络模块，F是常用函数（如Softmax）
import torch
import torch.nn as nn
import torch.nn.functional as F


class MyAtt(nn.Module):
    """
    【新手友好版】加性注意力（Bahdanau Attention）实现
    核心逻辑：拼接Query和Key → 非线性映射 → 打分 → Softmax权重 → 加权Value → 融合输出
    对应公式：score = v_a^T * tanh(W * concat(q, k))
    """

    def __init__(self, query_size, key_size, value_dim, attn_hidden_dim, output_size):
        """
        参数说明（全中文+维度解释）：
        - query_size: 解码器隐状态维度（Q的维度），比如128/256
        - key_size: 编码器隐状态维度（K的维度），通常和Q维度一致
        - value_dim: Value的维度（和K维度完全一致，因为V就是编码器隐状态）
        - attn_hidden_dim: 注意力隐藏层维度（公式中tanh后的维度），建议设为Q维度的一半
        - output_size: 最终输出维度（解码器下一步的输入维度）
        """
        # 继承nn.Module的初始化方法（必须写）
        super(MyAtt, self).__init__()

        # 保存参数（方便后续查看，新手可忽略）
        self.query_size = query_size
        self.key_size = key_size
        self.value_dim = value_dim
        self.attn_hidden_dim = attn_hidden_dim
        self.output_size = output_size

        # ------------------- 核心层定义 -------------------
        # 1. 拼接Q+K后的映射层：把(query_size+key_size)维映射到attn_hidden_dim维
        #    对应公式中的 W(concat(q,k)) 部分
        self.attn_cat = nn.Linear(self.query_size + self.key_size, self.attn_hidden_dim)

        # 2. 注意力打分层：把attn_hidden_dim维压缩成1维（标量得分） bias=false表示不需要计算偏置b
        self.attn_score = nn.Linear(self.attn_hidden_dim, 1, bias=False)

        # 3. 输出融合层：把(Q + 上下文向量)映射到最终输出维度
        #    作用：融合解码器当前状态和注意力上下文，给解码器下一步用
        self.attn_combine = nn.Linear(self.query_size + self.value_dim, self.output_size)

    def forward(self, Q, K, V):
        """
        前向传播（注意力计算的核心步骤）
        输入说明（新手重点记！）：
        - Q: 解码器当前步隐状态 → shape=[batch_size, query_size]
             比如 batch_size=2（2个句子）, query_size=128 → shape=[2, 128]
        - K: 编码器所有步隐状态 → shape=[batch_size, seq_len, key_size]
             比如 seq_len=3（句子有3个词） → shape=[2, 3, 128]
        - V: 编码器所有步隐状态（和K完全一样）→ shape=[batch_size, seq_len, value_dim]
        返回值：
        - output: 融合后的输出 → shape=[1, batch_size, output_size]
        - attn_weights: 注意力权重 → shape=[batch_size, seq_len]
        """
        # ------------------- 步骤1：获取基础维度信息 -------------------
        # batch_size：批次大小（有多少个句子）
        # seq_len：编码器序列长度（句子有多少个词）
        # _：占位符，忽略最后一个维度（key_size）
        batch_size, seq_len, _ = K.shape
        print(f"【调试】batch_size={batch_size}, seq_len={seq_len}")  # 新手可看维度变化

        # ------------------- 步骤2：扩展Q的维度（新手核心难点） -------------------
        # Q原始shape：[batch_size, query_size] → [2, 128]
        # 目标：让Q和K的维度匹配，能拼接 → 需要变成 [batch_size, seq_len, query_size] → [2, 3, 128]
        # unsqueeze(1)：在第1维加一个维度 → [2, 1, 128]
        # repeat(1, seq_len, 1)：第0维不变，第1维重复seq_len次，第2维不变 → [2, 3, 128]
        Q_expand = Q.unsqueeze(1).repeat(1, seq_len, 1)
        print(f"【调试】Q扩展后维度：{Q_expand.shape}")  # 新手可验证维度

        # ------------------- 步骤3：拼接Q和K -------------------
        # Q_expand: [2, 3, 128], K: [2, 3, 128]
        # dim=-1：在最后一维拼接 → [2, 3, 128+128=256]
        qk_cat = torch.cat((Q_expand, K), dim=-1)
        print(f"【调试】Q+K拼接后维度：{qk_cat.shape}")  # [2, 3, 256]

        # ------------------- 步骤4：计算注意力得分（核心公式） -------------------
        # 4.1 非线性变换：tanh是加性注意力的标配，增加非线性表达能力
        #     对应公式中的 tanh(W*concat(q,k)) 部分
        attn_hidden = torch.tanh(self.attn_cat(qk_cat))
        print(f"【调试】tanh后维度：{attn_hidden.shape}")  # [2, 3, 64]（假设attn_hidden_dim=64）

        # 4.2 压缩成标量得分：[2, 3, 64] → [2, 3, 1]
        attn_scores = self.attn_score(attn_hidden)
        # 4.3 挤压最后一维：[2, 3, 1] → [2, 3]（方便后续Softmax）
        attn_scores = attn_scores.squeeze(-1)
        print(f"【调试】注意力得分维度：{attn_scores.shape}")  # [2, 3]

        # ------------------- 步骤5：Softmax归一化（得到权重） -------------------
        # dim=-1：对最后一维（seq_len维度）做归一化，权重之和=1
        # 比如某一行：[0.2, 0.5, 0.3] → 表示对3个词的关注程度
        attn_weights = F.softmax(attn_scores, dim=-1)
        print(f"【调试】注意力权重维度：{attn_weights.shape}")  # [2, 3]
        print(f"【调试】权重之和（验证）：{attn_weights.sum(dim=-1)}")  # 应该≈1.0

        # ------------------- 步骤6：加权求和V（得到上下文向量） -------------------
        # 6.1 扩展权重维度：[2, 3] → [2, 1, 3]（适配bmm矩阵乘法）
        attn_weights_expand = attn_weights.unsqueeze(1)
        # 6.2 批量矩阵乘法（bmm）：权重 × V
        #     attn_weights_expand: [2, 1, 3] × V: [2, 3, 128] → [2, 1, 128]
        #     作用：对每个词的V按权重加权求和，得到上下文向量
        attn_applied = torch.bmm(attn_weights_expand, V)
        print(f"【调试】加权后V的维度：{attn_applied.shape}")  # [2, 1, 128]

        # ------------------- 步骤7：融合Q和上下文向量 -------------------
        # 7.1 挤压上下文向量维度：[2, 1, 128] → [2, 128]
        attn_applied_squeeze = attn_applied.squeeze(1)
        # 7.2 拼接Q和上下文向量：[2, 128] + [2, 128] → [2, 256]
        output_cat = torch.cat((Q, attn_applied_squeeze), dim=-1)
        # 7.3 映射到输出维度：[2, 256] → [2, output_size]（比如output_size=256）
        output = self.attn_combine(output_cat)
        # 7.4 扩展维度：[2, 256] → [1, 2, 256]（适配解码器输入格式）
        output = output.unsqueeze(0)
        print(f"【调试】最终输出维度：{output.shape}")  # [1, 2, 256]

        # 返回最终结果
        return output, attn_weights


# ------------------- 新手专用测试代码 -------------------
if __name__ == "__main__":
    # 【第一步】定义参数（新手可修改数值，观察维度变化）
    query_size = 128  # 解码器隐状态维度
    key_size = 128  # 编码器隐状态维度
    value_dim = 128  # V的维度（和K一致）
    attn_hidden_dim = 64  # 注意力隐藏层维度（建议是query_size的一半）
    output_size = 256  # 最终输出维度

    # 【第二步】初始化注意力层
    attn_layer = MyAtt(query_size, key_size, value_dim, attn_hidden_dim, output_size)
    print("=" * 50 + " 初始化完成 " + "=" * 50)

    # 【第三步】模拟输入（新手重点看维度！）
    batch_size = 2  # 2个句子
    seq_len = 3  # 每个句子3个词
    Q = torch.randn(batch_size, query_size)  # 解码器隐状态：[2, 128]
    K = torch.randn(batch_size, seq_len, key_size)  # 编码器隐状态：[2, 3, 128]
    V = torch.randn(batch_size, seq_len, value_dim)  # V和K一样：[2, 3, 128]
    print(f"\n【输入维度】Q:{Q.shape}, K:{K.shape}, V:{V.shape}")

    # 【第四步】前向计算
    print("\n" + "=" * 50 + " 开始计算注意力 " + "=" * 50)
    output, attn_weights = attn_layer(Q, K, V)

    # 【第五步】输出结果（验证是否正确）
    print("\n" + "=" * 50 + " 最终结果 " + "=" * 50)
    print(f"1. 注意力权重（每行和为1）：\n{attn_weights}")
    print(f"2. 注意力权重维度：{attn_weights.shape}")
    print(f"3. 最终输出维度：{output.shape}")

================================================== 初始化完成 ==================================================

【输入维度】Q:torch.Size([2, 128]), K:torch.Size([2, 3, 128]), V:torch.Size([2, 3, 128])

================================================== 开始计算注意力 ==================================================
【调试】batch_size=2, seq_len=3
【调试】Q扩展后维度：torch.Size([2, 3, 128])
【调试】Q+K拼接后维度：torch.Size([2, 3, 256])
【调试】tanh后维度：torch.Size([2, 3, 64])
【调试】注意力得分维度：torch.Size([2, 3])
【调试】注意力权重维度：torch.Size([2, 3])
【调试】权重之和（验证）：tensor([1., 1.], grad_fn=<SumBackward1>)
【调试】加权后V的维度：torch.Size([2, 1, 128])
【调试】最终输出维度：torch.Size([1, 2, 256])

================================================== 最终结果 ==================================================
1. 注意力权重（每行和为1）：
tensor([[0.3347, 0.3282, 0.3372],
        [0.3531, 0.3157, 0.3313]], grad_fn=<SoftmaxBackward0>)
2. 注意力权重维度：torch.Size([2, 3])
3. 最终输出维度：torch.Size([1, 2, 256])


上面的这个代码可能初步理解起来有点难度 可以先看下面的一个简化版本的代码加深理解, 在下面的代码中省略了特征降维和tanh的处理过程：

<img src="./markdown-note/images/img_37.png" width="700"/>

In [12]:
class MyAttSimple(nn.Module):
    def __init__(self, query_size, key_size, value_size1, value_size2, output_size):
        super(MyAttSimple, self).__init__()
        self.query_size = query_size
        self.key_size = key_size
        self.value_size1 = value_size1
        self.value_size2 = value_size2
        self.output_size = output_size
        # Q和K需要做拼接  所以线性层的输入维度是query_size+key_size
        self.attn = nn.Linear(self.query_size + self.key_size, self.value_size1)
        # 用于拼接 Q和上下文向量
        self.attn_combine = nn.Linear(self.query_size + self.value_size2, self.output_size)

    def forward(self, Q, K, V):
        # Q[0], K[0] 都是 [1, 32] cat之后 [1, 64],经过线性层处理之后变为[1, 32]，再经过softmax处理还是[1,32]
        attn_weights = F.softmax(self.attn(torch.cat((Q[0], K[0]), dim=-1)), dim=-1)
        # 将权重和V相乘 得到上下文向量
        # attn_weights 经过unsqueeze(0) 之后变为  [1, 1, 32] 和我们的 V (1,32,64)做加权和得到(1,1,64)
        attn_applied = torch.bmm(attn_weights.unsqueeze(0), V)
        # 拼接 Q和上下文向量 [1,32] 和 [1,64]拼接得到 [1,96]
        output = torch.cat((Q[0], attn_applied[0]), dim=-1)
        # 对Q和上下文向量的拼接结果进行线性层处理  [1,96] 经过线性层处理得到[1,32]，再经过 unsqueeze(0) 得到[1,1,32]
        output = self.attn_combine(output).unsqueeze(0)
        return output, attn_weights

Q = torch.randn(1, 1, 32)
K = torch.randn(1, 1, 32)
V = torch.randn(1, 32, 64)
attn_layer = MyAttSimple(query_size=32, key_size=32, value_size1=32, value_size2=64, output_size=32)
output, attn_weights = attn_layer(Q, K, V)
print(output.shape)
print(attn_weights.shape)



torch.Size([1, 1, 32])
torch.Size([1, 32])
